In [31]:
# 1. Install necessary tools
!pip install fastapi uvicorn nest-asyncio pyngrok joblib pandas

import nest_asyncio
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn


/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Server.serve' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [32]:
# 3. Create the App
app = FastAPI()

In [33]:
# Load the model
model = joblib.load('model.pkl')

In [34]:
from pydantic import Field, BaseModel
from typing import Annotated

class PredictionInput(BaseModel):
    age: Annotated[int , Field(..., description="Age of the customer", examples=[30])]
    gender: Annotated[str , Field(..., description="Gender of the customer (e.g., 'Male', 'Female')", examples=["Male"])]
    city: Annotated[str , Field(..., description="City where the customer resides", examples=["hydrabad"])]
    income: Annotated[float , Field(..., description="Annual income of the customer", examples=["10000"])]
    spending_score: Annotated[float ,Field(..., description="Spending score of the customer (0-10)", examples=[4.2])]
    membership_years:Annotated[float , Field(..., description="Number of years the customer has been a member", examples=[10])]

In [35]:
@app.get('/')
def index():
    return {'message': 'FastAPI is running on Colab!'}

@app.post('/predict')
def predict(data: PredictionInput):
    df = pd.DataFrame([data.dict()])
    prediction = model.predict(df)

    # Map 0 to 'No' and 1 to 'Yes'
    prediction_text = 'Yes you are recommended' if prediction[0] == 1 else 'No you are not recommended'

    return {
        'api_version': '1.0',
        'timestamp': pd.Timestamp.now().isoformat(),
        'recommendation': f'Based on the provided data, the customer recommendation is: {prediction_text}.',
        'friendly_response': {
            'raw_prediction': int(prediction[0]),
            'recommendation_status': prediction_text
        }
    }

In [36]:
# 4. Tunnel and Run
auth_token = "3DG4XjucHDN8im3Bxbp9chVaSdK_2pezrYoLgi5buaUWiYRXn"
ngrok.set_auth_token(auth_token)


In [37]:

# Create tunnel
public_url = ngrok.connect(8000)
print(f"--- API is live at: {public_url} ---")
print("Append /docs to the URL above to test it!")
nest_asyncio.apply()


--- API is live at: NgrokTunnel: "https://citizen-uncle-stillness.ngrok-free.dev" -> "http://localhost:8000" ---
Append /docs to the URL above to test it!


In [38]:
import threading
def run_uvicorn():
    uvicorn.run(app, port=8000)

thread = threading.Thread(target=run_uvicorn)
thread.start()

INFO:     Started server process [5486]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
